In [0]:
# =============================================================
# Notebook : 03_gold_inventory_health.py
# Purpose  : Inventory health dashboard data
# Source   : walmart_silver.fact_inventory_daily
# Target   : gold/gold_inventory_health/
# =============================================================

GOLD_PATH = "abfss://gold@walmartdata.dfs.core.windows.net/"

df_inv_health = spark.sql("""
    SELECT
        snapshot_date,
        store_id,
        store_city,
        store_region,
        category,
        supplier_id,
        COUNT(DISTINCT sku)                 AS total_skus,
        SUM(closing_stock)                  AS total_units_on_hand,
        ROUND(SUM(inventory_value), 2)      AS total_inventory_value,
        COUNT(CASE WHEN needs_reorder
              THEN 1 END)                   AS skus_to_reorder,
        COUNT(CASE WHEN stockout_risk = 'STOCKOUT'
              THEN 1 END)                   AS stockout_skus,
        COUNT(CASE WHEN stockout_risk = 'CRITICAL'
              THEN 1 END)                   AS critical_skus,
        ROUND(AVG(sellthrough_rate), 2)     AS avg_sellthrough_rate,
        ROUND(AVG(days_of_stock_remaining), 1) AS avg_days_stock,
        current_timestamp()                 AS gold_processed_at
    FROM walmart_silver.fact_inventory_daily
    GROUP BY
        snapshot_date, store_id, store_city,
        store_region, category, supplier_id
    ORDER BY stockout_skus DESC, critical_skus DESC
""")

(
    df_inv_health
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("snapshot_date", "store_region")
    .save(f"{GOLD_PATH}gold_inventory_health/")
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_gold.gold_inventory_health
    USING DELTA
    LOCATION '{GOLD_PATH}gold_inventory_health/'
""")

print("✅ walmart_gold.gold_inventory_health registered")

spark.sql("""
    SELECT store_region, category,
           SUM(stockout_skus)  AS stockouts,
           SUM(critical_skus)  AS critical,
           SUM(skus_to_reorder) AS needs_reorder
    FROM walmart_gold.gold_inventory_health
    GROUP BY store_region, category
    ORDER BY stockouts DESC
""").display()